In [ ]:
#imports 
import csv
from pathlib import Path
import pandas as pd
import numpy as np
import os
from scipy.stats import spearmanr, ttest_rel, f_oneway, linregress
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import math
import pingouin as pg
import random
from datetime import datetime
from scipy.signal import resample
from scipy.interpolate import interp1d
import matplotlib as mpl

In [ ]:
# Gets all day directories (like D01, D02, etc.) for a specific subject.
# Filters folder names that begin with "D", sorts them, and returns the list of paths.
def get_day_paths(subject_path: Path): # checked
    days = os.listdir(subject_path)
    day_paths = sorted(list(map(lambda x: subject_path / x, list(filter(lambda x: x[0] == "D", days)))))

    return day_paths

In [ ]:
# Reads the interpolated.csv file inside the given day/segment folder 
# Returns a dictionary of DataFrames with keys: "affected finger", "affected wrist"
def read_assigned_sensor_csv(day_path: Path): # checked
    """
    Read all sensor data. Concatenate and order if there are more than one file.
    """
    assigned_data: pd.DataFrame = pd.read_csv(list(day_path.glob("assigned.csv"))[0])

    affected_finger: pd.DataFrame = assigned_data["affected finger"]
    affected_wrist: pd.DataFrame = assigned_data["affected wrist"]

    return {"affected finger": affected_finger,
            "affected wrist": affected_wrist}

In [ ]:
# Gets all subject directories inside the main data folder, ignoring hidden files
# For each subject folder:
    # Calls get_day_paths() to get all day paths.
    # Stores them in a dictionary where the key is the subject name and the value is a list of day paths.
def get_subject_day_paths(data_path: Path) -> dict[str, list[Path]]: # checked
    all_subject_dir_names: list[str] = os.listdir(data_path)
    all_subject_paths: list[Path] = sorted(list(
        map(lambda x: data_path / x, list(filter(lambda x: x[0] != ".", all_subject_dir_names)))
    ))
    subject_day_paths = {}

    for subject_path in all_subject_paths:
        subject = subject_path.name
        subject_day_paths[subject] = get_day_paths(subject_path=subject_path)

    return subject_day_paths

In [ ]:
# Prepares the output folder structure to store processed assign-combined files.
def create_dirs(valid_subject_day_paths: dict[str, list[Path]], output_dir: Path): # checked
    subjects: list[str] = valid_subject_day_paths.keys()

    for subject in subjects:
        subject_output_path: Path = output_dir / subject
        if not os.path.exists(subject_output_path):
            os.makedirs(subject_output_path)

        day_paths: list[Path] = valid_subject_day_paths[subject]
        days: list[str] = list(map(lambda day_path: day_path.name[:6], day_paths))

        for day in days:
            day_output_path: Path = output_dir / subject / day
            if not os.path.exists(day_output_path):
                os.makedirs(day_output_path)

In [ ]:
def combine_segments(assigned_dir: Path, output_dir: Path):
    """
    Combine segmented 'assigned.csv' files (if present) for each subject/day into one final CSV per day.

    Args:
        assigned_dir (Path): The directory containing the assigned data (with potential subdirectories for each subject/day).
        output_dir (Path): The directory where the combined CSV files will be saved.
    """

    # Step 1: Get mapping of subjects to their corresponding list of day directories.
    # e.g., {"Subject01": [Path(".../Subject01/Day1"), Path(".../Subject01/Day2")], ...}
    valid_subject_day_paths: dict[str, list[Path]] = get_subject_day_paths(data_path=assigned_dir)

    # Step 2: Make sure the output directories exist for each subject/day pair.
    # This prepares the folder structure in `output_dir` so we can save combined CSVs later.
    create_dirs(valid_subject_day_paths=valid_subject_day_paths, output_dir=output_dir)

    # Step 3: Extract subject IDs (keys of the dict)
    subjects: list[str] = valid_subject_day_paths.keys()

    # Step 4: Iterate over each subject and their corresponding day folders
    for subject in subjects:
        day_paths = valid_subject_day_paths[subject]

        for day_path in day_paths:
            # Print for progress/debugging
            print(subject, day_path.name)

            # List all files and folders in that day's directory
            files_in_day_path: list[str] = os.listdir(day_path)

            # --- Case 1: The day folder contains a single 'assigned.csv' (no segments)
            if "segment1" not in files_in_day_path:
                # Read that assigned.csv
                df_all: pd.DataFrame = pd.read_csv(day_path / "assigned.csv")

                # Save it directly to the corresponding path in output_dir
                df_all.to_csv(output_dir / subject / day_path.name / "assigned.csv", index=False)

            # --- Case 2: The day folder is divided into multiple 'segment' folders (e.g., segment1, segment2, ...)
            elif "segment1" in files_in_day_path:
                # Collect all segment folder paths in sorted order (important to preserve sequence)
                segment_paths: list[Path] = sorted(list(day_path.glob("segment*")))
                print(segment_paths)

                # Create an empty DataFrame to accumulate data
                df_all: pd.DataFrame = pd.DataFrame({})

                # Loop through each segment directory
                for segment_path in segment_paths:
                    # Each segment has its own assigned.csv — read it
                    df_segment: pd.DataFrame = pd.read_csv(segment_path / "assigned.csv")

                    # Concatenate this segment’s data with the main combined DataFrame
                    df_all = pd.concat([df_all, df_segment])

                # Finally, save the combined DataFrame to output_dir
                df_all.to_csv(output_dir / subject / day_path.name / "assigned.csv", index=False)

In [ ]:
combine_segments(assigned_dir=Path("path/to/your/data"), output_dir=Path("path/to/your/data"))